In [4]:
import sqlite3
import pandas as pd

# Load the cleaned dataset
df = pd.read_csv('online_retail_cleaned.csv')

# Connect to (or create) the SQLite database file
conn = sqlite3.connect('online_retail.db')

# Write the dataframe into a SQL table
df.to_sql('OnlineRetail', conn, if_exists='replace', index=False)

print("Database created successfully!")
print(f"Table has {len(df)} rows")

Database created successfully!
Table has 392692 rows


In [5]:
# Query 1: Top 10 best-selling products (by quantity)
query1 = """
SELECT 
    Description,
    SUM(Quantity) AS TotalQuantitySold
FROM OnlineRetail
GROUP BY Description
ORDER BY TotalQuantitySold DESC
LIMIT 10
"""
top_products = pd.read_sql(query1, conn)
print(top_products)

                          Description  TotalQuantitySold
0         PAPER CRAFT , LITTLE BIRDIE              80995
1      MEDIUM CERAMIC TOP STORAGE JAR              77916
2   WORLD WAR 2 GLIDERS ASSTD DESIGNS              54319
3             JUMBO BAG RED RETROSPOT              46078
4  WHITE HANGING HEART T-LIGHT HOLDER              36706
5       ASSORTED COLOUR BIRD ORNAMENT              35263
6     PACK OF 72 RETROSPOT CAKE CASES              33670
7                      POPCORN HOLDER              30919
8                  RABBIT NIGHT LIGHT              27153
9             MINI PAINT SET VINTAGE               26076


In [6]:
# Query 2: Top 10 countries by revenue
query2 = """
SELECT 
    Country,
    SUM(TotalPrice) AS TotalRevenue
FROM OnlineRetail
GROUP BY Country
ORDER BY TotalRevenue DESC
LIMIT 10
"""
top_countries = pd.read_sql(query2, conn)
print(top_countries)

          Country  TotalRevenue
0  United Kingdom   7285024.644
1     Netherlands    285446.340
2            EIRE    265262.460
3         Germany    228678.400
4          France    208934.310
5       Australia    138453.810
6           Spain     61558.560
7     Switzerland     56443.950
8         Belgium     41196.340
9          Sweden     38367.830


In [7]:
# Query 3: Sales by month (seasonal analysis)
query3 = """
SELECT 
    Month,
    SUM(TotalPrice) AS MonthlyRevenue,
    COUNT(DISTINCT InvoiceNo) AS NumberOfOrders
FROM OnlineRetail
GROUP BY Month
ORDER BY Month
"""
monthly_sales = pd.read_sql(query3, conn)
print(monthly_sales)

    Month  MonthlyRevenue  NumberOfOrders
0       1      568101.310             987
1       2      446084.920             997
2       3      594081.760            1321
3       4      468374.331            1149
4       5      677355.150            1555
5       6      660046.050            1393
6       7      598962.901            1331
7       8      644051.040            1280
8       9      950690.202            1755
9      10     1035642.450            1929
10     11     1156205.610            2657
11     12     1087613.170            2178


In [8]:
# Query 4: Top 10 spending customers
query4 = """
SELECT 
    CustomerID,
    SUM(TotalPrice) AS TotalSpent,
    COUNT(DISTINCT InvoiceNo) AS NumberOfOrders
FROM OnlineRetail
GROUP BY CustomerID
ORDER BY TotalSpent DESC
LIMIT 10
"""
top_customers = pd.read_sql(query4, conn)
print(top_customers)

   CustomerID  TotalSpent  NumberOfOrders
0       14646   280206.02              73
1       18102   259657.30              60
2       17450   194390.79              46
3       16446   168472.50               2
4       14911   143711.17             201
5       12415   124914.53              21
6       14156   117210.08              55
7       17511    91062.38              31
8       16029    80850.84              63
9       12346    77183.60               1


In [9]:
# Check for extreme outliers in Quantity
query_outliers = """
SELECT *
FROM OnlineRetail
WHERE Quantity > 5000
ORDER BY Quantity DESC
"""
outliers = pd.read_sql(query_outliers, conn)
print(outliers)

# Flag extreme outlier orders (single bulk purchases that skew customer-level analysis)
# InvoiceNo is stored as integer in the dataframe, so use integer values here
outlier_invoices = [581483, 541431]

# Create a version of the data excluding these outliers
df_no_outliers = df[~df['InvoiceNo'].isin(outlier_invoices)]

print(f"Rows before removing outliers: {len(df)}")
print(f"Rows after removing outliers: {len(df_no_outliers)}")

# Save the outlier-free version as a separate table for customer-behavior analysis
df_no_outliers.to_sql('OnlineRetail_NoOutliers', conn, if_exists='replace', index=False)
print("Outlier-free table created successfully!")

   InvoiceNo StockCode                     Description  Quantity  \
0     581483     23843     PAPER CRAFT , LITTLE BIRDIE     80995   
1     541431     23166  MEDIUM CERAMIC TOP STORAGE JAR     74215   

           InvoiceDate  UnitPrice  CustomerID         Country  TotalPrice  \
0  2011-12-09 09:15:00       2.08       16446  United Kingdom    168469.6   
1  2011-01-18 10:01:00       1.04       12346  United Kingdom     77183.6   

   Month  Year DayOfWeek  
0     12  2011    Friday  
1      1  2011   Tuesday  
Rows before removing outliers: 392692
Rows after removing outliers: 392690
Outlier-free table created successfully!
